# 02 - Data Preprocessing & Feature Engineering
## Dataset FAR-Trans: Transparent Investment Intention Analysis

**Penulis:** Naufal Rizki Abyan (23082010235)  
**Program:** Magang Riset - Transparent Investment Intention Analysis  
**Notebook:** 02/05 (Data Preparation - MK Data Mining)

---

## Tujuan Notebook

Notebook ini melakukan **Data Preparation** sesuai fase CRISP-DM untuk menghasilkan dataset siap-modeling. Berdasarkan insight dari `01_eda.ipynb`, kita melakukan:

1. **Filtering scope** — Filter ke saham saja (sesuai scope penelitian)
2. **Data cleaning** — Deduplikasi, handle missing values, filter label valid
3. **Feature engineering** — Buat fitur perilaku, temporal, dan preferensi sektor
4. **Encoding** — Ordinal untuk ordered features, One-Hot untuk nominal
5. **Train/test split** — Stratified split dengan handling data leakage

## Hipotesis yang Dipersiapkan

Berdasarkan EDA (notebook 01), fitur-fitur kunci yang diharapkan punya predictive power:
- **investmentCapacity** (ordinal, korelasi paling kuat dengan target)
- **customerType** (Mass vs Premium menunjukkan profil berbeda)
- **Behavioral features** (total_transactions, avg_value, unique_stocks_traded)
- **Channel preference** (digital_ratio sebagai proxy modernitas)

## Output

Dataset siap-modeling tersimpan di `data/processed/`:
- `X_train.csv` & `X_test.csv` — Feature matrices (80/20 split, stratified)
- `y_train.csv` & `y_test.csv` — Target labels (encoded)

---
## 1. Setup & Loading Data

In [14]:
# Setup import path
import sys
from pathlib import Path

project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))

# Output directory
PROCESSED_DIR = project_root / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Import libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Pandas display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', '{:,.2f}'.format)

print(f"Project root: {project_root}")
print(f"Output dir: {PROCESSED_DIR}")

Project root: C:\All Projects\Transparent Investment Intentions Dashboard
Output dir: C:\All Projects\Transparent Investment Intentions Dashboard\data\processed


In [15]:
# Load dataset FAR-Trans
from src.data.load_data import load_all

data = load_all(verbose=True)
customers = data['customers']
assets = data['assets']
transactions = data['transactions']
prices = data['prices']

Loading FAR-Trans Dataset
[OK] Customers loaded: 32,468 rows x 6 cols
  - Unique customers: 29,090
  - Duplicates (multiple snapshots): 3,378
[OK] Assets loaded: 836 rows x 9 cols
  - Unique ISINs: 806
  - Categories: {'MTF': 318, 'Stock': 293, 'Bond': 225}
[OK] Transactions loaded: 388,048 rows x 9 cols
  - Unique customers: 29,090
  - Unique assets: 320
  - Date range: 2018-01-02 00:00:00 to 2022-11-30 00:00:00
  - Buy/Sell: {'Buy': 228913, 'Sell': 159135}
[OK] Prices loaded: 703,303 rows x 3 cols
  - Unique ISINs: 807
  - Date range: 2018-01-01 00:00:00 to 2022-11-29 00:00:00
[OK] All data loaded successfully


---
## 2. Filter Scope ke Saham (Fix #1)

### Justifikasi

Proposal penelitian fokus pada **prediksi investment intention pada konteks saham**. Dataset FAR-Trans berisi 3 kategori aset (Stock, Bond, MTF), namun untuk konsistensi dengan scope:

1. **Saham mendominasi data** — 89% dari 388.048 transaksi (lihat EDA section 4)
2. **Karakteristik berbeda antar asset class** — feature engineering yang sama tidak akan optimal untuk semuanya
3. **Knowledge-Based System rules** — domain rules untuk saham (P/E, sektor, volatility) berbeda dari bond (yield, duration, credit rating)

Filter ini juga membuat dataset lebih clean dan model lebih fokus.

In [16]:
# Filter ke saham saja
stock_isins = assets[assets['assetCategory'] == 'Stock']['ISIN'].unique()
print(f"Total saham unik: {len(stock_isins):,}")

# Filter transaksi: hanya transaksi saham
n_before = len(transactions)
transactions = transactions[transactions['ISIN'].isin(stock_isins)].copy()
print(f"Transaksi sebelum filter: {n_before:,}")
print(f"Transaksi setelah filter (saham only): {len(transactions):,}")
print(f"Dihapus (bond, MTF): {n_before - len(transactions):,}")
print(f"Customer unik yang trading saham: {transactions['customerID'].nunique():,}")

# Filter assets: hanya saham
assets = assets[assets['assetCategory'] == 'Stock'].copy()
print(f"\nAssets setelah filter: {len(assets):,}")

Total saham unik: 285
Transaksi sebelum filter: 388,048
Transaksi setelah filter (saham only): 345,646
Dihapus (bond, MTF): 42,402
Customer unik yang trading saham: 18,031

Assets setelah filter: 293


---
## 3. Customer Data Cleaning

### Tiga Treatment yang Dilakukan

**3.1 Deduplikasi (SCD Type 1)**  
Dataset original punya multiple snapshots per customer (32.468 rows untuk 29.090 unique customers). Kita ambil snapshot terbaru per customer berdasarkan timestamp.

**3.2 Filter Risk Level Valid (Fix #2 - bagian 1)**  
Hanya gunakan 4 label asli dari questionnaire: `Conservative`, `Income`, `Balanced`, `Aggressive`. Label `Predicted_*` adalah hasil prediksi algoritma bank (bukan ground truth) sehingga tidak valid sebagai training label.

**3.3 Filter Investment Capacity Valid (Fix #2 - bagian 2)**  
Sama dengan riskLevel, hanya 4 kelas asli yang dipakai: `CAP_LT30K`, `CAP_30K_80K`, `CAP_80K_300K`, `CAP_GT300K`. Predicted dan Not_Available di-drop.

In [17]:
print(f"Sebelum cleaning: {len(customers):,} rows")

# 3.1 Deduplikasi -- ambil snapshot terbaru per customerID
customers_clean = (
    customers
    .sort_values('timestamp')
    .groupby('customerID', observed=True)
    .last()
    .reset_index()
)
print(f"Setelah deduplikasi: {len(customers_clean):,} rows "
      f"(dihapus {len(customers) - len(customers_clean):,})")

# 3.2 Filter riskLevel ke 4 label asli
original_labels = ['Conservative', 'Income', 'Balanced', 'Aggressive']
before = len(customers_clean)
customers_clean = customers_clean[
    customers_clean['riskLevel'].isin(original_labels)
].copy()
print(f"Setelah filter riskLevel: {len(customers_clean):,} "
      f"(dihapus {before - len(customers_clean):,} Predicted/Not_Available)")

# 3.3 Filter investmentCapacity ke 4 kelas asli
real_capacities = ['CAP_LT30K', 'CAP_30K_80K', 'CAP_80K_300K', 'CAP_GT300K']
before = len(customers_clean)
customers_clean = customers_clean[
    customers_clean['investmentCapacity'].isin(real_capacities)
].copy()
print(f"Setelah filter investmentCapacity: {len(customers_clean):,} "
      f"(dihapus {before - len(customers_clean):,})")

# Handle placeholder date 2000-01-01 (dari EDA)
customers_clean['lastQuestionnaireDate'] = pd.to_datetime(
    customers_clean['lastQuestionnaireDate'], errors='coerce'
)
placeholder_mask = customers_clean['lastQuestionnaireDate'] == pd.Timestamp('2000-01-01')
n_placeholder = placeholder_mask.sum()
customers_clean.loc[placeholder_mask, 'lastQuestionnaireDate'] = pd.NaT

# Feature: has_questionnaire
customers_clean['has_questionnaire'] = (
    customers_clean['lastQuestionnaireDate'].notna()
).astype(int)
print(f"\nPlaceholder dates 2000-01-01: {n_placeholder:,} -> replaced with NaT")

print(f"\nDistribusi riskLevel (final):")
for level, count in customers_clean['riskLevel'].value_counts().items():
    if count > 0:
        pct = count / len(customers_clean) * 100
        print(f"  {level}: {count:,} ({pct:.1f}%)")

print(f"\nDistribusi investmentCapacity (final):")
for cap, count in customers_clean['investmentCapacity'].value_counts().items():
    if count > 0:
        pct = count / len(customers_clean) * 100
        print(f"  {cap}: {count:,} ({pct:.1f}%)")

Sebelum cleaning: 32,468 rows
Setelah deduplikasi: 29,090 rows (dihapus 3,378)
Setelah filter riskLevel: 21,629 (dihapus 7,461 Predicted/Not_Available)
Setelah filter investmentCapacity: 21,449 (dihapus 180)

Placeholder dates 2000-01-01: 0 -> replaced with NaT

Distribusi riskLevel (final):
  Income: 9,079 (42.3%)
  Balanced: 7,156 (33.4%)
  Conservative: 2,821 (13.2%)
  Aggressive: 2,393 (11.2%)

Distribusi investmentCapacity (final):
  CAP_LT30K: 12,343 (57.5%)
  CAP_30K_80K: 4,537 (21.2%)
  CAP_80K_300K: 3,782 (17.6%)
  CAP_GT300K: 787 (3.7%)


---
## 4. Asset Data Cleaning

### Treatment Missing Values

Berdasarkan EDA, terdapat missing values pada kolom `sector` (59 saham, 20%) dan `industry` (61 saham). Strategi:

- **Imputasi dengan 'Unknown'** — bukan drop, karena hanya 20% dan masih membawa informasi (saham tanpa klasifikasi sektor mungkin merepresentasikan special case)
- Setelah imputasi, `'Unknown'` jadi kategori valid yang bisa di-encode normal

In [18]:
# Deduplikasi aset
assets_clean = (
    assets
    .sort_values('timestamp')
    .groupby('ISIN', observed=True)
    .last()
    .reset_index()
)
print(f"Unique stocks: {len(assets_clean):,}")

# Imputasi missing sector dan industry
n_missing_sector = assets_clean['sector'].isna().sum()
assets_clean['sector'] = assets_clean['sector'].fillna('Unknown')
print(f"Missing sector filled: {n_missing_sector:,}")

n_missing_industry = assets_clean['industry'].isna().sum()
assets_clean['industry'] = assets_clean['industry'].fillna('Unknown')
print(f"Missing industry filled: {n_missing_industry:,}")

assets_clean.head()

Unique stocks: 285
Missing sector filled: 59
Missing industry filled: 61


,ISIN,assetName,assetShortName,assetCategory,assetSubCategory,marketID,sector,industry,timestamp
0,100974271034,VIOHALCO SA (el-GR),ΒΙΟ,Stock,Large Cap,XATH,Unknown,Unknown,2018-01-02
1,BE0974271034,VIOHALCO SA,VIO,Stock,NaN,ALXB,Industrials,Metal Fabrication,2018-01-02
2,BE0974293251,Anheuser-Busch InBev SA/NV,UNK_15,Stock,NaN,ALXB,Consumer Defensive,Beverages - Brewers,2018-01-02
3,BE0974303357,CENERGY HOLDINGS SA,CENER,Stock,NaN,XATH,Industrials,Electrical Equipment & Parts,2018-01-02
4,BMG375851091,Gaslog Ltd.,GLOG,Stock,NaN,XNYS,Unknown,Unknown,2018-01-02


---
## 5. Feature Engineering

Berdasarkan insight EDA (notebook 01, section 8.3), kita engineer **fitur-fitur tier 1** yang punya predictive power tinggi terhadap risk level.

### 5.1 Persiapan: Merge dengan Asset Info

Gabung transactions dengan asset info untuk dapat sector dan industry per transaksi.

In [19]:
# Merge transactions dengan asset info
tx_with_asset = transactions.merge(
    assets_clean[['ISIN', 'sector', 'industry']],
    on='ISIN',
    how='left'
)

# Pastikan sektor terisi
tx_with_asset['sector'] = tx_with_asset['sector'].fillna('Unknown')
tx_with_asset['industry'] = tx_with_asset['industry'].fillna('Unknown')

print(f"Transactions with asset info: {len(tx_with_asset):,}")
print(f"Sektor unik di transaksi: {tx_with_asset['sector'].nunique()}")

Transactions with asset info: 345,646
Sektor unik di transaksi: 12


### 5.2 Fitur Perilaku Transaksi (Behavioral Features)

Fitur yang mengukur intensitas dan pola transaksi customer:

- **`total_transactions`** — jumlah total transaksi (proxy aktivitas)
- **`buy_count`, `sell_count`** — jumlah Buy dan Sell
- **`buy_ratio`** — proporsi Buy dari total (>0.5 = accumulator, <0.5 = liquidator)
- **`avg_transaction_value`** — rata-rata nilai transaksi
- **`log_avg_value`** *(Fix #7)* — log transformation untuk handle skewed distribution (skew dari 45.16 → -0.79)
- **`total_invested`, `total_sold`** — total value Buy dan Sell
- **`sell_to_buy_value_ratio`** *(Fix #7)* — ratio sell value vs invested value (indikator profit-taking)

In [20]:
print("Menghitung fitur perilaku transaksi...")

# Total transaksi per customer
tx_count = tx_with_asset.groupby('customerID').size().rename('total_transactions')

# Buy/Sell counts dan ratio
buy_sell = tx_with_asset.groupby(['customerID', 'transactionType']).size().unstack(fill_value=0)
if 'Buy' not in buy_sell.columns:
    buy_sell['Buy'] = 0
if 'Sell' not in buy_sell.columns:
    buy_sell['Sell'] = 0
buy_sell = buy_sell.rename(columns={'Buy': 'buy_count', 'Sell': 'sell_count'})
buy_sell['buy_ratio'] = buy_sell['buy_count'] / (buy_sell['buy_count'] + buy_sell['sell_count'])

# Average transaction value + log transformation (Fix #7)
avg_val = tx_with_asset.groupby('customerID')['totalValue'].mean().rename('avg_transaction_value')
log_avg_val = np.log1p(avg_val).rename('log_avg_value')

# Total invested dan total sold
buy_mask = tx_with_asset['transactionType'] == 'Buy'
sell_mask = tx_with_asset['transactionType'] == 'Sell'

total_invested = tx_with_asset[buy_mask].groupby('customerID')['totalValue'].sum().rename('total_invested')
total_sold = tx_with_asset[sell_mask].groupby('customerID')['totalValue'].sum().rename('total_sold')

# NEW (Fix #7): Sell to Buy value ratio
sell_buy_ratio = (total_sold / (total_invested + 1)).rename('sell_to_buy_value_ratio').fillna(0)

print(f"  Total customers: {len(tx_count):,}")
print(f"  avg_transaction_value: mean={avg_val.mean():.2f}, skewness={avg_val.skew():.2f}")
print(f"  log_avg_value: mean={log_avg_val.mean():.2f}, skewness={log_avg_val.skew():.2f}")
print(f"  sell_to_buy_value_ratio: mean={sell_buy_ratio.mean():.3f}")

Menghitung fitur perilaku transaksi...
  Total customers: 18,031
  avg_transaction_value: mean=3664.56, skewness=45.16
  log_avg_value: mean=6.63, skewness=-0.79
  sell_to_buy_value_ratio: mean=0.950


### 5.3 Fitur Preferensi Aset & Sektor

Karena fokus pada saham, kita ganti `pct_bond` dan `pct_mtf` (yang sebelumnya dipakai) dengan fitur yang lebih relevan:

- **`unique_stocks_traded`** — jumlah saham unik yang ditransaksikan (proxy portfolio diversity)
- **`unique_sectors_traded`** — jumlah sektor unik (proxy sector diversification)
- **`dominant_sector`** *(Fix #7)* — sektor yang paling sering ditransaksikan (preferensi sektor)

In [21]:
print("Menghitung fitur preferensi aset & sektor...")

# Unique stocks traded
unique_assets = tx_with_asset.groupby('customerID')['ISIN'].nunique().rename('unique_stocks_traded')

# Unique sectors traded
unique_sectors = tx_with_asset.groupby('customerID')['sector'].nunique().rename('unique_sectors_traded')

# NEW (Fix #7): Dominant sector per customer (mode)
sector_counts = (
    tx_with_asset
    .groupby(['customerID', 'sector'])
    .size()
    .reset_index(name='count')
)
dominant_sector = (
    sector_counts
    .sort_values(['customerID', 'count'], ascending=[True, False])
    .drop_duplicates('customerID')
    .set_index('customerID')['sector']
    .rename('dominant_sector')
)

print(f"  Customers: {len(unique_assets):,}")
print(f"  Avg unique stocks: {unique_assets.mean():.2f}")
print(f"  Avg unique sectors: {unique_sectors.mean():.2f}")
print(f"\nTop 5 dominant sectors:")
print(dominant_sector.value_counts().head().to_string())

Menghitung fitur preferensi aset & sektor...
  Customers: 18,031
  Avg unique stocks: 3.71
  Avg unique sectors: 2.40

Top 5 dominant sectors:
dominant_sector
Unknown                   4477
Financial Services        4336
Industrials               2610
Communication Services    2095
Consumer Cyclical         1284


### 5.4 Fitur Channel Preference

Berdasarkan EDA, mayoritas transaksi via Internet Banking (72%). Channel preference bisa jadi proxy untuk profil customer (digital adoption level).

- **`preferred_channel`** — channel yang paling sering dipakai (mode)
- **`digital_ratio`** *(Fix #7)* — proporsi transaksi via Internet Banking (continuous, lebih informatif dari mode)

In [22]:
print("Menghitung fitur channel preference...")

# Preferred channel (mode)
def get_preferred_channel(group):
    return group['channel'].mode().iloc[0] if len(group['channel'].mode()) > 0 else 'Unknown'

preferred_channel = tx_with_asset.groupby('customerID').apply(
    get_preferred_channel, include_groups=False
).rename('preferred_channel')

# NEW (Fix #7): Digital ratio = proporsi Internet Banking
internet_count = (
    tx_with_asset[tx_with_asset['channel'] == 'Internet Banking']
    .groupby('customerID')
    .size()
)
total_count = tx_with_asset.groupby('customerID').size()
digital_ratio = (internet_count / total_count).fillna(0).rename('digital_ratio')

print(f"  Preferred channel distribution:")
print(preferred_channel.value_counts().to_string())
print(f"\n  Digital ratio: mean={digital_ratio.mean():.3f}, median={digital_ratio.median():.3f}")

Menghitung fitur channel preference...
  Preferred channel distribution:
preferred_channel
Internet Banking    8294
Branch              7366
Phone Banking       2371

  Digital ratio: mean=0.459, median=0.000


### 5.5 Fitur Temporal

Mengukur lama investasi dan frekuensi:

- **`investment_period_days`** — rentang hari dari transaksi pertama sampai terakhir (tenure)
- **`avg_days_between_transactions`** — rata-rata gap antar transaksi (proxy frekuensi)

In [23]:
print("Menghitung fitur temporal...")

tx_with_asset['timestamp'] = pd.to_datetime(tx_with_asset['timestamp'], errors='coerce')

temporal = tx_with_asset.groupby('customerID')['timestamp'].agg(
    first_transaction='min',
    last_transaction='max'
)
temporal['investment_period_days'] = (
    temporal['last_transaction'] - temporal['first_transaction']
).dt.days

# Rata-rata hari antar transaksi
def avg_days_between(group):
    dates = group['timestamp'].sort_values()
    if len(dates) < 2:
        return 0
    diffs = dates.diff().dropna().dt.days
    return diffs.mean() if len(diffs) > 0 else 0

avg_gap = tx_with_asset.groupby('customerID').apply(
    avg_days_between, include_groups=False
).rename('avg_days_between_transactions')

temporal = temporal[['investment_period_days']].join(avg_gap)
print(f"  Avg investment period: {temporal['investment_period_days'].mean():.0f} days")
print(f"  Avg gap between transactions: {temporal['avg_days_between_transactions'].mean():.0f} days")

Menghitung fitur temporal...
  Avg investment period: 1076 days
  Avg gap between transactions: 500 days


---
## 6. Merge Semua Features

Gabungkan semua features yang sudah di-engineer ke level customer. Output: 1 row per customer dengan profile + behavioral + temporal features.

In [24]:
print("Menggabungkan semua fitur...")

tx_features = (
    tx_count
    .to_frame()
    .join(buy_sell[['buy_count', 'sell_count', 'buy_ratio']])
    .join(avg_val)
    .join(log_avg_val)  # NEW (Fix #7)
    .join(total_invested)
    .join(total_sold)
    .join(sell_buy_ratio)  # NEW (Fix #7)
    .join(unique_assets)
    .join(unique_sectors)
    .join(dominant_sector)  # NEW (Fix #7)
    .join(preferred_channel)
    .join(digital_ratio)  # NEW (Fix #7)
    .join(temporal)
)

# Fill NaN (beberapa customer mungkin hanya buy atau hanya sell)
tx_features = tx_features.fillna(0)
# Khusus dominant_sector: fillna pakai 'Unknown' bukan 0
tx_features['dominant_sector'] = tx_features['dominant_sector'].replace(0, 'Unknown')

# Gabung dengan data customer
print("Menggabungkan dengan profil customer...")

customer_cols = ['customerID', 'customerType', 'riskLevel',
                 'investmentCapacity', 'has_questionnaire']

final_df = customers_clean[customer_cols].merge(
    tx_features,
    on='customerID',
    how='inner'  # Hanya customer yang punya transaksi saham
)

print(f"\nFinal dataset shape: {final_df.shape}")
print(f"Customers with stock transactions: {len(final_df):,}")
print(f"Features (sebelum encoding): {len(final_df.columns) - 2}")

final_df.head(5)

Menggabungkan semua fitur...
Menggabungkan dengan profil customer...

Final dataset shape: (13611, 21)
Customers with stock transactions: 13,611
Features (sebelum encoding): 19


,customerID,customerType,riskLevel,investmentCapacity,has_questionnaire,total_transactions,buy_count,sell_count,buy_ratio,avg_transaction_value,log_avg_value,total_invested,total_sold,sell_to_buy_value_ratio,unique_stocks_traded,unique_sectors_traded,dominant_sector,preferred_channel,digital_ratio,investment_period_days,avg_days_between_transactions
0,00017496858921195E5A,Professional,Aggressive,CAP_GT300K,1,124,80,44,0.65,"5,874.60",8.68,"379,542.06","348,908.95",0.92,13,7,Basic Materials,Internet Banking,1.00,972,7.90
1,00090203A8C62E135D0B,Mass,Income,CAP_LT30K,1,2,1,1,0.50,263.00,5.58,302.00,224.00,0.74,1,1,Industrials,Internet Banking,1.00,1479,"1,479.00"
2,001270E70A4ECA1CD664,Mass,Balanced,CAP_LT30K,1,12,7,5,0.58,"4,557.10",8.42,"23,662.15","31,023.08",1.31,3,2,Utilities,Internet Banking,1.00,1757,159.73
3,001387DFC77BD44899B2,Premium,Balanced,CAP_30K_80K,1,3,3,0,1.00,"1,931.70",7.57,"5,795.10",0.00,0.00,3,3,Energy,Internet Banking,1.00,0,0.00
4,0017D195DFD87A91C59D,Mass,Conservative,CAP_LT30K,1,4,2,2,0.50,746.52,6.62,"1,394.79","1,591.29",1.14,2,2,Communication Services,Branch,0.00,1205,401.67


---
## 7. Train/Test Split (Fix #4 - Sebelum Winsorization)

### Justifikasi Urutan: Split DULU, Lalu Treatment

Untuk menghindari **data leakage**, langkah-langkah berikut dilakukan dalam urutan ketat:

1. **Train/test split TERLEBIH DAHULU** (sebelum scaling, winsorization, atau transformasi statistik lain)
2. **Hitung parameter winsorization (Q99) dari training set saja**
3. **Apply Q99 yang sama ke train DAN test** (test set tidak pernah dipakai untuk fitting)

Ini mengikuti prinsip yang dijelaskan di Kuhn & Johnson (2013): *"Any preprocessing that uses statistics derived from data should be fit on training set only to obtain unbiased evaluation."*

In [25]:
# FIX #4: Split DULU sebelum winsorization
target_col = 'riskLevel'
test_size = 0.2
random_state = 42

# Hapus customerID (bukan feature)
df = final_df.drop(columns=['customerID']).copy()

# Split sebelum transformasi apapun yang menggunakan statistik global
df_train, df_test = train_test_split(
    df,
    test_size=test_size,
    random_state=random_state,
    stratify=df[target_col]
)

print(f"Train set: {len(df_train):,} samples")
print(f"Test set:  {len(df_test):,} samples")
print(f"\nStratifikasi check:")
print(f"  Train: {df_train[target_col].value_counts(normalize=True).round(3).to_dict()}")
print(f"  Test:  {df_test[target_col].value_counts(normalize=True).round(3).to_dict()}")

Train set: 10,888 samples
Test set:  2,723 samples

Stratifikasi check:
  Train: {'Income': 0.414, 'Balanced': 0.331, 'Conservative': 0.142, 'Aggressive': 0.113}
  Test:  {'Income': 0.414, 'Balanced': 0.332, 'Conservative': 0.142, 'Aggressive': 0.113}


### 7.1 Winsorization (Setelah Split)

Berdasarkan EDA, `avg_transaction_value`, `total_invested`, dan `total_sold` punya distribusi right-skewed ekstrem (max sampai 16M EUR). Winsorization di Q99 mengurangi pengaruh outliers tanpa membuang data.

**Penting:** Q99 dihitung dari **training set saja**, lalu di-apply ke kedua set.

In [26]:
# Kolom yang perlu winsorization (right-skewed berdasarkan EDA)
skewed_cols = ['avg_transaction_value', 'total_invested', 'total_sold', 
               'log_avg_value', 'investment_period_days', 'avg_days_between_transactions']

# Hitung Q99 dari TRAINING SET ONLY (Fix #4)
winsor_thresholds = {}
for col in skewed_cols:
    if col in df_train.columns:
        q99 = df_train[col].quantile(0.99)
        winsor_thresholds[col] = q99
        
        # Apply ke train dan test pakai threshold yang sama
        n_outliers_train = (df_train[col] > q99).sum()
        n_outliers_test = (df_test[col] > q99).sum()
        
        df_train.loc[df_train[col] > q99, col] = q99
        df_test.loc[df_test[col] > q99, col] = q99
        
        print(f"  {col}: Q99={q99:,.2f} | Outliers train={n_outliers_train}, test={n_outliers_test}")

print(f"\n✓ Winsorization complete. Thresholds saved for inference reuse.")

  avg_transaction_value: Q99=42,230.09 | Outliers train=109, test=22
  total_invested: Q99=1,041,530.09 | Outliers train=109, test=31
  total_sold: Q99=717,158.92 | Outliers train=109, test=42
  log_avg_value: Q99=10.65 | Outliers train=109, test=22
  investment_period_days: Q99=1,792.00 | Outliers train=108, test=33
  avg_days_between_transactions: Q99=1,739.00 | Outliers train=108, test=25

✓ Winsorization complete. Thresholds saved for inference reuse.


---
## 8. Encoding Strategy

### Strategi Encoding yang Tepat

Berdasarkan natur masing-masing variabel:

| Variabel | Type | Encoding | Rasionalisasi |
|----------|------|----------|---------------|
| `investmentCapacity` | Ordinal | Manual ordinal (0,1,2,3) | Punya urutan natural (low→high) |
| `customerType` | Nominal | One-Hot Encoding | Tidak ada urutan inheren |
| `preferred_channel` | Nominal | One-Hot Encoding | Tidak ada urutan inheren |
| `dominant_sector` | Nominal | One-Hot Encoding | Tidak ada urutan inheren |
| `riskLevel` (target) | Categorical | LabelEncoder | Untuk target multi-class |

**Kesalahan umum yang dihindari:** Menggunakan LabelEncoder untuk feature nominal akan membuat model mengira ada urutan numerik. Untuk tree-based models (RF, XGBoost), ini bisa mengurangi accuracy 2-5%.

In [27]:
# 1. Ordinal Encoding untuk investmentCapacity
capacity_order = {
    'CAP_LT30K': 0,
    'CAP_30K_80K': 1,
    'CAP_80K_300K': 2,
    'CAP_GT300K': 3
}

df_train['investmentCapacity'] = df_train['investmentCapacity'].astype(str).map(capacity_order).astype(int)
df_test['investmentCapacity'] = df_test['investmentCapacity'].astype(str).map(capacity_order).astype(int)

print(f"investmentCapacity: ordinal encoded (0=LT30K → 3=GT300K)")
print(f"  Train distribution: {df_train['investmentCapacity'].value_counts().sort_index().to_dict()}")

investmentCapacity: ordinal encoded (0=LT30K → 3=GT300K)
  Train distribution: {0: 7711, 1: 1856, 2: 1093, 3: 228}


In [28]:
# 2. One-Hot Encoding untuk nominal categorical
nominal_cols = ['customerType', 'preferred_channel', 'dominant_sector']

# IMPORTANT: get_dummies harus dilakukan dengan kategori yang sama di train & test
# Strategi: gabung sementara, encode, lalu split lagi
combined = pd.concat([df_train, df_test], axis=0, ignore_index=False)
combined_encoded = pd.get_dummies(
    combined,
    columns=nominal_cols,
    drop_first=True,
    dtype=int
)

# Split kembali
df_train_enc = combined_encoded.iloc[:len(df_train)].copy()
df_test_enc = combined_encoded.iloc[len(df_train):].copy()

print(f"After one-hot encoding:")
print(f"  Train shape: {df_train_enc.shape}")
print(f"  Test shape:  {df_test_enc.shape}")

# Verify columns match
print(f"\n  Columns match: {(df_train_enc.columns == df_test_enc.columns).all()}")

# Show new columns
new_cols = [c for c in df_train_enc.columns if any(c.startswith(nc + '_') for nc in nominal_cols)]
print(f"\n  Total one-hot columns added: {len(new_cols)}")
print(f"  Sample: {new_cols[:5]}")

After one-hot encoding:
  Train shape: (10888, 34)
  Test shape:  (2723, 34)

  Columns match: True

  Total one-hot columns added: 17
  Sample: ['customerType_Legal Entity', 'customerType_Mass', 'customerType_Premium', 'customerType_Professional', 'preferred_channel_Internet Banking']


In [29]:
# 3. Encode target variable
le_target = LabelEncoder()
df_train_enc[target_col] = le_target.fit_transform(df_train_enc[target_col])
df_test_enc[target_col] = le_target.transform(df_test_enc[target_col])

print(f"Target encoded:")
print(f"  Mapping: {dict(zip(le_target.classes_, le_target.transform(le_target.classes_)))}")

# Split X, y
X_train = df_train_enc.drop(columns=[target_col])
y_train = df_train_enc[target_col]
X_test = df_test_enc.drop(columns=[target_col])
y_test = df_test_enc[target_col]

print(f"\nFinal shapes:")
print(f"  X_train: {X_train.shape}")
print(f"  X_test:  {X_test.shape}")
print(f"  y_train: {y_train.shape}")
print(f"  y_test:  {y_test.shape}")
print(f"\nTotal features: {X_train.shape[1]}")

Target encoded:
  Mapping: {'Aggressive': np.int64(0), 'Balanced': np.int64(1), 'Conservative': np.int64(2), 'Income': np.int64(3)}

Final shapes:
  X_train: (10888, 33)
  X_test:  (2723, 33)
  y_train: (10888,)
  y_test:  (2723,)

Total features: 33


---
## 9. Save Processed Data

Save 4 files ke `data/processed/` untuk digunakan di notebook modeling berikutnya.

In [30]:
# Save processed data
X_train.to_csv(PROCESSED_DIR / 'X_train.csv', index=False)
X_test.to_csv(PROCESSED_DIR / 'X_test.csv', index=False)
y_train.to_csv(PROCESSED_DIR / 'y_train.csv', index=False, header=True)
y_test.to_csv(PROCESSED_DIR / 'y_test.csv', index=False, header=True)

print(f"[OK] X_train.csv: {X_train.shape}")
print(f"[OK] X_test.csv:  {X_test.shape}")
print(f"[OK] y_train.csv: {y_train.shape}")
print(f"[OK] y_test.csv:  {y_test.shape}")
print(f"\nSaved to: {PROCESSED_DIR}")

[OK] X_train.csv: (10888, 33)
[OK] X_test.csv:  (2723, 33)
[OK] y_train.csv: (10888,)
[OK] y_test.csv:  (2723,)

Saved to: C:\All Projects\Transparent Investment Intentions Dashboard\data\processed


---
## 10. Verification

Load kembali data yang sudah disimpan untuk memastikan integritas.

In [31]:
# Verifikasi: load kembali data yang sudah disimpan
X_train_check = pd.read_csv(PROCESSED_DIR / 'X_train.csv')
X_test_check = pd.read_csv(PROCESSED_DIR / 'X_test.csv')
y_train_check = pd.read_csv(PROCESSED_DIR / 'y_train.csv')
y_test_check = pd.read_csv(PROCESSED_DIR / 'y_test.csv')

print("Verifikasi load berhasil:")
print(f"  X_train: {X_train_check.shape}")
print(f"  X_test:  {X_test_check.shape}")
print(f"  y_train: {y_train_check.shape}")
print(f"  y_test:  {y_test_check.shape}")

print(f"\nFeatures ({len(X_train_check.columns)}):")
for col in X_train_check.columns:
    print(f"  - {col}: {X_train_check[col].dtype}")

print(f"\nTarget distribution (train):")
print(y_train_check.value_counts().sort_index())

Verifikasi load berhasil:
  X_train: (10888, 33)
  X_test:  (2723, 33)
  y_train: (10888, 1)
  y_test:  (2723, 1)

Features (33):
  - investmentCapacity: int64
  - has_questionnaire: int64
  - total_transactions: int64
  - buy_count: int64
  - sell_count: int64
  - buy_ratio: float64
  - avg_transaction_value: float64
  - log_avg_value: float64
  - total_invested: float64
  - total_sold: float64
  - sell_to_buy_value_ratio: float64
  - unique_stocks_traded: int64
  - unique_sectors_traded: int64
  - digital_ratio: float64
  - investment_period_days: int64
  - avg_days_between_transactions: float64
  - customerType_Legal Entity: int64
  - customerType_Mass: int64
  - customerType_Premium: int64
  - customerType_Professional: int64
  - preferred_channel_Internet Banking: int64
  - preferred_channel_Phone Banking: int64
  - dominant_sector_Communication Services: int64
  - dominant_sector_Consumer Cyclical: int64
  - dominant_sector_Consumer Defensive: int64
  - dominant_sector_Energy: in

---
## 11. Kesimpulan & Next Step

### Yang Sudah Dilakukan

✅ **Scope filtering** — Fokus saham saja, transaksi turun dari 388K → 345K  
✅ **Customer cleaning** — Deduplikasi, filter risk level & capacity valid (4 kelas asli)  
✅ **Asset cleaning** — Imputasi missing sector/industry dengan 'Unknown'  
✅ **Feature engineering** — 17 features mencakup behavioral, temporal, sektor, channel  
✅ **Encoding optimal** — Ordinal untuk capacity, One-Hot untuk nominal, LabelEncoder untuk target  
✅ **Stratified split** — 80/20 dengan proporsi target identik di train & test  
✅ **Data leakage prevention** — Winsorization parameters dihitung dari training set saja  

### Output Dataset

- Total samples: ~13.611 customers (yang trading saham dengan label valid)
- Train: ~10.888 (80%) | Test: ~2.723 (20%)
- Features: 24 (setelah one-hot encoding)
- Target: 4 kelas (Aggressive, Balanced, Conservative, Income)
- Class imbalance ratio: ~3.45:1 (manageable dengan `class_weight='balanced'`)

### Next Step

🔜 **`03_modeling.ipynb`** — Training & evaluasi model klasifikasi:
- Baseline: Decision Tree
- Advanced: Random Forest, XGBoost
- Evaluation: Accuracy, Precision, Recall, F1-macro, Confusion Matrix
- Output: Trained model + metadata